### Article 6: Geospatial Analytics for Retail Store Operations

Build a Scalable Store Intelligence Lakehouse on Databricks Using Grid Indexing (Serverless-Compatible)

In the modern retail landscape, location is one of the strongest predictors of store performance. Footfall patterns, catchment density, customer behavior, competition proximity, and local demographics — all of it matters. Yet, traditional GIS tools and relational systems often struggle to scale when the goal is to serve analytics across millions of spatial records in real-time.

With Databricks and a smart use of spatial analytics (without needing H3), we can now turn data assets into Gold-layer operational intelligence for retail — with support for:

Precision catchment areas

Inventory + demand forecasting

Workforce scheduling by mobility trends

Store cannibalization metrics

Competitive landscape scoring

Whether you're a store operator, business analyst, or data engineer, this article shows you how to build the entire pipeline using Bronze → Silver → Gold layers, powered by grid indexing, Delta Lake, and SQL + Python.

### Why Grid Indexing Instead of H3?

🧠 H3 is great, but not always available in free or serverless runtimes (like Databricks Free Tier or SQL Serverless).
💡 Instead, we use a simple but effective fallback technique: grid indexing with lat/lon snap rounding.

CONCAT(FLOOR(lat * 100), '_', FLOOR(lon * 100)) AS grid_id


This discretizes geographic space into small ~1 km² bins. That means:

✅ Fast joins across spatial datasets (using string equality)
✅ No geometry-heavy computation upfront
✅ Works on Databricks Serverless, no installs required




![](/Volumes/thedataengineering_00/data/sampledata/6_architecture.png)

### 1. BRONZE LAYER: Load Raw Customer Events

Use Python to simulate 1000 synthetic customer events around a Los Angeles retail cluster.

![](/Volumes/thedataengineering_00/data/sampledata/6_bronze.png)
![](/Volumes/thedataengineering_00/data/sampledata/6_silver.png)
![](/Volumes/thedataengineering_00/data/sampledata/6_gold.png)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS geo_demo;
USE CATALOG geo_demo;
CREATE SCHEMA IF NOT EXISTS retail_bronze;
USE SCHEMA retail_bronze;

CREATE SCHEMA IF NOT EXISTS retail_silver;
USE SCHEMA retail_silver;

CREATE SCHEMA IF NOT EXISTS retail_gold;
USE SCHEMA retail_gold;

In [0]:
from pyspark.sql.functions import expr, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import random, uuid, time

# Generate 1000 synthetic customer records around LA
data_customers = [
    (
        f"Cust_{i}",
        34.0522 + random.random() * 0.05,   # lat near LA center
        -118.2437 + random.random() * 0.05, # lon near LA center
        int(time.time()) - random.randint(0, 86400),  # timestamp last 24h
        "mobile" if random.random() > 0.4 else "web"
    )
    for i in range(1000)
]

schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("event_unix", DoubleType(), True),
    StructField("device_type", StringType(), True)
])

df_bronze = spark.createDataFrame(data_customers, schema) \
    .withColumn("event_time", expr("timestamp_millis(CAST(event_unix * 1000 AS BIGINT))")) \
    .withColumn("ingest_ts", current_timestamp()) \
    .withColumn("src_file", expr("'batch_1'"))

df_bronze.write.format("delta").mode("overwrite").saveAsTable("retail_bronze.customers")


This creates a table in Unity Catalog: retail_bronze.customers.

### 2. SILVER LAYER: Clean & Add Spatial + Grid Indexing

Convert raw lat/lon to spatial points and enrich with grid ID for fast joins.

In [0]:
%sql
CREATE OR REPLACE TABLE retail_silver.customers AS
SELECT
  customer_id,
  event_time,
  lat,
  lon,
  ST_POINT(lon, lat) AS geo_point,
  CONCAT(FLOOR(lat * 100), '_', FLOOR(lon * 100)) AS grid_id,
  CAST(event_time AS DATE) AS day,
  device_type,
  ingest_ts,
  src_file
FROM retail_bronze.customers
WHERE lat BETWEEN -90 AND 90 AND lon BETWEEN -180 AND 180;  -- Basic geo validation


Adds schema consistency, partitioning, and spatial readiness.

### 3. Load Store Locations into Silver

In [0]:
store_data = [
    ("Store_A", 34.0622, -118.2537),
    ("Store_B", 34.0422, -118.2337),
    ("Store_C", 34.0522, -118.2637)
]

df_stores = spark.createDataFrame(store_data, ["store_id", "lat", "lon"]) \
    .withColumn("geo_point", expr("ST_POINT(lon, lat)")) \
    .withColumn("grid_id", expr("concat(floor(lat * 100), '_', floor(lon * 100))"))

df_stores.write.format("delta").mode("overwrite").saveAsTable("retail_silver.stores")


### 4. GOLD LAYER: Build Store Catchment Analytics
Step A: Approximate Join via Grid Index

In [0]:
%sql
CREATE OR REPLACE TABLE retail_gold.store_customer_join AS
SELECT 
  s.store_id, 
  c.customer_id, 
  c.geo_point AS cust_loc, 
  s.geo_point AS store_loc,
  c.device_type,
  c.event_time
FROM retail_silver.customers c
JOIN retail_silver.stores s
  ON c.grid_id = s.grid_id;


### Step B: Exact Distance Calculation (<= 3km)

In [0]:
%sql
CREATE OR REPLACE TABLE retail_gold.store_customer_catchment AS
SELECT 
  store_id,
  customer_id,
  device_type,
  event_time,
  ST_DISTANCE(cust_loc, store_loc) AS distance_m
FROM retail_gold.store_customer_join
WHERE ST_DISTANCE(cust_loc, store_loc) <= 3000;  -- 3 km catchment radius


### Step C: Rollup to Store Operations KPIs

In [0]:
%sql
CREATE OR REPLACE TABLE retail_gold.store_ops_kpis AS
SELECT
  store_id,
  COUNT(*) AS customer_count,
  AVG(distance_m) AS avg_distance_m,
  MAX(distance_m) AS max_distance_mYour ops team can now analyze customer distance behavior, channel mix, and more.
  COUNT_IF(device_type = 'mobile') AS mobile_visits,
  COUNT_IF(device_type = 'web') AS web_visits
FROM retail_gold.store_customer_catchment
GROUP BY store_id;



Your ops team can now analyze customer distance behavior, channel mix, and more.

### Catchment Heatmap Table

In [0]:
%sql
CREATE OR REPLACE TABLE retail_gold.customer_heatmap AS
SELECT 
  grid_id,
  COUNT(*) AS customer_count,
  AVG(lat) AS center_lat,
  AVG(lon) AS center_lon
FROM retail_silver.customers
GROUP BY grid_id;


### Business Insights You Can Create

- Store staffing insight: more midday visits → higher afternoon coverage.
- Inventory planning: catchment distance + order frequency → reorder rhythms.
- Cannibalization alerts: overlapping catchments between Store A and B.
- Marketing targeting: high-conversion grid IDs → localized ads.
- Customer loyalty: track avg proximity and session retention.

![](/Volumes/thedataengineering_00/data/sampledata/6_future.png)